In [ ]:
# Practice Problems Day 15

### Author: Joanna Bieri (solutions)

## Introduction to Machine Learning

#### University of Redlands - DATA 301
#### Prof: Joanna Bieri [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
#### Day 15 Practice - SOLUTIONS
#### [Class Website](https://joannabieri.com/machine_learning.html)

---

**These are practice problems. They are not graded and you do not turn them in.** They are here so you can practice today's code: an embedding, an RNN, and writing your own network as a class. Plan on about 45 minutes. Each training run takes less than a minute on a laptop.

**These are my solutions.** Try every problem yourself first, then come here to check your work. There is more than one right way to write the code, and your words will be different from mine, that is ok! If your numbers do not match mine, look for the difference before you look at my code.

**Reading:** Geron chapter 13, the sections **LSTMs** and **GRUs**, and chapter 14, the section **Embeddings** (inside "Generating Shakespearean Text Using a Character RNN").

GOALS:

1. Write a network as a class with `__init__` and `forward`, using `nn.Embedding` and `nn.RNN`.
2. Compare an RNN and a GRU on long reviews.
3. Explain in words why an RNN can not learn from the start of a long review.

**How much should you write yourself?** The first code cell is given: the words, the review maker and the training function from the notes. In Problem 1 most of the class is there and you write the missing lines. Problem 2 lists the steps and you write the code. Problem 3 is words only. The **Optional exploration** at the end is extra, only if you are curious.

**Save your work.** Commit and push when you are done, so you have your code for later:

```bash
git add .
git commit -m "Day 15 practice"
git push
```

---

In [ ]:
# RUN THIS. The words, the review maker and the training function from the notes.
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

feeling_words = ["great", "good", "fun", "loved", "wonderful",       # positive, ids 0 to 4
                 "bad", "boring", "awful", "hated", "terrible"]      # negative, ids 5 to 9
filler_words = ["the", "movie", "was", "and", "a", "plot", "actors", "it", "story", "long",
                "music", "ending", "i", "saw", "with", "my", "friends", "on", "friday", "popcorn"]
vocab = feeling_words + filler_words      # every word gets an id: its spot in this list
vocab_size = len(vocab)

def make_reviews(n_reviews, n_words, seed):
    generator = torch.Generator().manual_seed(seed)       # so we get the same reviews every time
    # Start with all filler words (ids 10 to 29)
    X = torch.randint(10, vocab_size, (n_reviews, n_words), generator=generator)
    # Pick one feeling word (ids 0 to 9) and one spot for it, for every review
    feeling = torch.randint(0, 10, (n_reviews,), generator=generator)
    spot = torch.randint(0, n_words, (n_reviews,), generator=generator)
    X[torch.arange(n_reviews), spot] = feeling           # drop the feeling word into its spot
    # Positive (ids 0 to 4) is label 1, negative (ids 5 to 9) is label 0
    y = (feeling < 5).long()
    return X, y

# The mini-batch training loop from the notes. It returns the validation accuracy after every epoch.
def train_model(model, X_train, y_train, X_valid, y_valid, n_epochs):
    torch.manual_seed(42)
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

    valid_accuracy = []
    for epoch in range(n_epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            scores = model(X_batch)
            loss = loss_fn(scores, y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            y_pred = model(X_valid).argmax(dim=1)
            accuracy = (y_pred == y_valid).float().mean().item()
        valid_accuracy.append(accuracy)
        print("epoch", epoch + 1, "  valid accuracy:", round(accuracy, 4))

    return valid_accuracy

# 20 word reviews for Problem 1
X_train, y_train = make_reviews(4000, n_words=20, seed=0)
X_valid, y_valid = make_reviews(1000, n_words=20, seed=1)
print("X_train:", X_train.shape)

## Problem 1: Write the RNN class

In the notes we wrote our first network as a class, `ReviewNet`. Now you get to write it yourself! Most of the class is below, and you fill in the four lines the comments ask for. Try it without looking at the notes first, then check.

This time the reviews are **20 words** long. That is in between the 10 word reviews (where the RNN got 1.0) and the 100 word reviews (where it stayed near 0.5) from the notes.

**1a.** Fill in the class and train it on the 20 word reviews for 5 epochs.

**1b.** Count how many numbers your network has to learn. Would that number change if the reviews were 100 words long? Why or why not?

**1c.** How does your accuracy compare to the notes? There the RNN got 1.0 on the 10 word reviews and stayed near 0.5 on the 100 word reviews.

In [ ]:
class ReviewNet(nn.Module):

    def __init__(self):
        super().__init__()                       # always the first line
        self.embed = nn.Embedding(vocab_size, 8)          # every word becomes 8 numbers
        self.memory = nn.RNN(8, 32, batch_first=True)     # 8 numbers in, a memory of 32 numbers
        self.out = nn.Linear(32, 2)                       # go from 32 to 2 scores: negative, positive

    def forward(self, X):
        vectors = self.embed(X)                           # X = (reviews, 20) -> vectors = (reviews, 20, 8)
        all_hidden, last_hidden = self.memory(vectors)    # read every review word by word - TWO outputs!
        scores = self.out(last_hidden[0])                 # [0] takes (1, reviews, 32) to (reviews, 32)
        return scores

torch.manual_seed(42)
model_1 = ReviewNet()
accuracy_1 = train_model(model_1, X_train, y_train, X_valid, y_valid, n_epochs=5)

In [ ]:
# 1b: count the numbers to learn, the same loop as in the notes
n_params = 0
for p in model_1.parameters():
    n_params = n_params + p.numel()
print("numbers to learn:", n_params)

**1a.** The four missing lines are `self.embed = nn.Embedding(vocab_size, 8)`, `self.memory = nn.RNN(8, 32, batch_first=True)`, `vectors = self.embed(X)` and `all_hidden, last_hidden = self.memory(vectors)`. After 5 epochs the validation accuracy is 1.0! It started at 0.54 and it took until epoch 4 to get there.

**1b.** 1,650 numbers: 240 in the embedding (30 words times 8 numbers), 1,344 in the RNN and 66 in the Linear layer. No, it would not change. The RNN uses the SAME weights for every word, so a longer review just means we use the same box more times. This is why one RNN can read a review of any length.

**1c.** It still gets to 1.0 on the 20 word reviews, but it took 4 epochs instead of 1. On the 10 word reviews it was 1.0 from the first epoch, and on the 100 word reviews it never got away from 0.5. So the longer the review, the harder it is for the RNN to learn.

---

## Problem 2: RNN or GRU on 50 word reviews (steps outlined)

Now let's make the reviews longer, 50 words, and see which network can still learn them. Copy your class from Problem 1 and write the steps below yourself.

**2a.** Make 50 word reviews (4,000 to train on with `seed=0`, 1,000 to validate on with `seed=1`). Train a new `ReviewNet` on them for 5 epochs. Then write a `ReviewGRU` class, the same as `ReviewNet` with `nn.GRU` in place of `nn.RNN`, and train it the same way.

**2b.** Which network learned the 50 word reviews? Use the words **gradient** and **gate** to explain why.

In [ ]:
# Step 1: the 50 word reviews
X_train_50, y_train_50 = make_reviews(4000, n_words=50, seed=0)
X_valid_50, y_valid_50 = make_reviews(1000, n_words=50, seed=1)

# Step 2: the RNN
torch.manual_seed(42)
rnn_50 = ReviewNet()
accuracy_rnn_50 = train_model(rnn_50, X_train_50, y_train_50, X_valid_50, y_valid_50, n_epochs=5)

In [ ]:
# Step 3: the GRU class, the same as ReviewNet with ONE change
class ReviewGRU(nn.Module):

    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, 8)          # every word becomes 8 numbers
        self.memory = nn.GRU(8, 32, batch_first=True)     # THE ONE CHANGE: a GRU instead of an RNN
        self.out = nn.Linear(32, 2)                       # 2 scores: negative, positive

    def forward(self, X):
        vectors = self.embed(X)
        all_hidden, last_hidden = self.memory(vectors)    # a GRU also gives back TWO things
        scores = self.out(last_hidden[0])
        return scores

# Step 4: train it
torch.manual_seed(42)
gru_50 = ReviewGRU()
accuracy_gru_50 = train_model(gru_50, X_train_50, y_train_50, X_valid_50, y_valid_50, n_epochs=5)

**2a.** The RNN never gets away from 0.5: 0.509, 0.477, 0.492, 0.504, 0.531. The GRU gets 1.0 in the very first epoch!

**2b.** The GRU. In the RNN the gradient gets multiplied by a small number at every word on the way back. If the feeling word is 30 or 40 words from the end, its gradient is close to 0 by the time it gets there, so the RNN can not learn from it. The GRU has an update gate. When the gate is close to 1 the GRU keeps its old memory and skips over the filler word, and then the gradient on the way back gets multiplied by about 1 at that step instead of something small. So the gradient does not vanish and the GRU can learn from the early words.

---

## Problem 3: In words

No code here, just explaining. Writing it out is the best way to check that you really understand it!

**3a.** Explain to a friend in DATA 101 why an RNN has trouble learning from the start of a long review. Use the words **hidden state** and **gradient**. Three or four sentences, no code.

**3b.** Here is a real sentence: *"The movie that my friends from high school, who I had not seen in years, picked for Friday night was great."* To know what was great, which two words does a network need to connect? How many words are between them? Why is that hard for an RNN?

**3a.** Your answer will be different, and that is ok! Here is one way to say it:

An RNN reads a review one word at a time and keeps a list of numbers, its hidden state, as its memory. Every new word gets mixed into that memory. When the network learns, the gradient has to go back through every one of those steps, and it gets multiplied by a small number each time. By the time it gets back to the first words it is almost 0 (in the notes the first 25 words got exactly 0), so the network can not learn anything from them.

**3b.** "movie" and "great". There are 18 words between them! The RNN has to carry "this is about the movie" in its memory through all 18 of those words, and the gradient that connects "great" back to "movie" has to go back through all 18 steps, getting smaller at every step. A GRU or LSTM has a better chance, and attention (Day 16) connects the two words directly.

---

## Optional exploration

Only if you are curious! Nothing here is needed later.

**A.** Where does the RNN break? Train `ReviewNet` on 30 and 40 word reviews. Where does the accuracy drop?

**B.** Try an LSTM, the older version of the GRU idea from the reading. Use `nn.LSTM(8, 32, batch_first=True)`. BEWARE: for the last memory it gives back a pair, so the forward line becomes `all_hidden, (last_hidden, last_cell) = self.memory(vectors)`. Train it on the 50 word reviews.

**C.** Play with the sizes! In your `ReviewGRU` class the embedding size is the 8 in `nn.Embedding(vocab_size, 8)` and in `nn.GRU(8, 32, batch_first=True)`. Change BOTH of them. Try an embedding size of 2, 8 and 32 on the 50 word reviews. For each one write down the validation accuracy after every epoch and how many numbers there are to learn (the `p.numel()` loop from Problem 1). Does a bigger embedding always help? You can also try the memory size (the 32, in two places), the learning rate (`lr=0.01` inside `train_model`) or the number of epochs.

NOTE: With an embedding size of 2 every word is just 2 numbers, so you could plot the words directly, no PCA needed!

In [ ]:
# A: 30 and 40 word reviews, one for loop so we do not copy the same code twice
for n_words in [30, 40]:
    print("reviews with", n_words, "words")
    X_tr, y_tr = make_reviews(4000, n_words=n_words, seed=0)
    X_va, y_va = make_reviews(1000, n_words=n_words, seed=1)
    torch.manual_seed(42)
    rnn_model = ReviewNet()
    accuracy = train_model(rnn_model, X_tr, y_tr, X_va, y_va, n_epochs=5)

In [ ]:
# B: an LSTM on the 50 word reviews
class ReviewLSTM(nn.Module):

    def __init__(self):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, 8)
        self.memory = nn.LSTM(8, 32, batch_first=True)    # an LSTM instead of a GRU
        self.out = nn.Linear(32, 2)

    def forward(self, X):
        vectors = self.embed(X)
        all_hidden, (last_hidden, last_cell) = self.memory(vectors)   # the LSTM gives back a pair for the last memory
        scores = self.out(last_hidden[0])
        return scores

torch.manual_seed(42)
lstm_50 = ReviewLSTM()
accuracy_lstm_50 = train_model(lstm_50, X_train_50, y_train_50, X_valid_50, y_valid_50, n_epochs=5)

In [ ]:
# C: the embedding size as an argument, so one for loop can try 2, 8 and 32
class ReviewGRUSize(nn.Module):

    def __init__(self, embed_size):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_size)         # every word becomes embed_size numbers
        self.memory = nn.GRU(embed_size, 32, batch_first=True)    # the GRU has to take in embed_size numbers too
        self.out = nn.Linear(32, 2)

    def forward(self, X):
        vectors = self.embed(X)
        all_hidden, last_hidden = self.memory(vectors)
        scores = self.out(last_hidden[0])
        return scores

for embed_size in [2, 8, 32]:
    torch.manual_seed(42)
    model = ReviewGRUSize(embed_size)
    n_params = 0
    for p in model.parameters():
        n_params = n_params + p.numel()
    print("embedding size", embed_size, "  numbers to learn:", n_params)
    accuracy = train_model(model, X_train_50, y_train_50, X_valid_50, y_valid_50, n_epochs=5)

**A.** On the 30 word reviews the RNN sits near 0.5 for four epochs and then jumps to 0.895 in epoch 5. On the 40 word reviews it is still at 0.522 after 5 epochs. So with 5 epochs it breaks somewhere between 30 and 40 words. With more epochs it might get further, try it!

**B.** The LSTM gets there too: 0.656, 0.849, 0.999, then 1.0. It is a little slower than the GRU here, which got 1.0 in the first epoch.

**C.** With an embedding size of 2 there are 3,582 numbers to learn and the GRU is slower: 0.461, 0.554, then 0.998 in epoch 3 and 1.0 after that. Size 8 (4,338 numbers, the one from the notes) gets 1.0 from the first epoch. Size 32 (7,362 numbers) gets 0.996 and then 1.0. So bigger is not always better! Our made up reviews only need to know if a word is positive, negative or filler, so a few numbers per word is enough. A real vocabulary, where words have lots of different meanings, needs a lot more.